In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()


True

<div style="font: bold 40px sans-serif;">LangChain Runtime 运行时机制</div>

<br/>
LangChain的Runtime机制是理解Agent内部运行状态的关键。它包括三个核心概念：


| 概念 | 说明                      | 生命周期 |
|-----|-------------------------|---------|
| **State** | 短期记忆，存储Agent当前对话信息、任务状态 | 单次请求 |
| **Store** | 长期记忆，包含用户偏好、失败经验等       | 跨会话 |
| **Context** | 运行时上下文，传递配置参数           | 单次请求 |


# 1. State（短期记忆）

State是Agent的短期记忆，存储当前会话的历史消息、任务状态等信息。

我们之前学习短期记忆时使用的是默认的AgentState，其中只包含会话的历史消息，本节开始我们学习如何自定义AgentState，记录除了历史消息以外的其他信息。


## 1.1 自定义State

In [2]:
from langchain.agents import AgentState
from typing import NotRequired


# 定义自定义State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: NotRequired[int]  # 模型调用次数
    session_start: NotRequired[str]  # 会话开始时间

D:\code\PycharmProjects\langchain_course\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 1.2.在工具中访问state

在定义tool的时候，LangChain内置了一个runtime参数，通过runtime我们可以获取Agent的内部信息，包括：
- state
- store
- context

因此，runtime成为了LangChain中tool的限定参数，自定义参数不能叫这个名字。


In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from datetime import datetime


@tool
def update_state(runtime: ToolRuntime):
    """A tool that update agent state"""
    # 获取state中的历史消息
    messages = runtime.state['messages']
    # 消息数量
    message_count = len(messages)
    # 组织结果
    command = {
        "model_call_count": runtime.state.get("model_call_count", 0) + 1,
        "messages": [ToolMessage("Successfully updated agent state", tool_call_id=runtime.tool_call_id)]
    }
    # 判断是否是第一次
    if message_count <= 2:
        command['session_start'] = datetime.now()

    return Command(update=command)

In [21]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "deepseek-v4-flash",
    tools=[update_state],
    state_schema=CustomState,
    checkpointer=InMemorySaver(),
    system_prompt="你是一个热心的助手，你必须在每次回答用户前调用update_state工具以更新任务状态。调用工具时不要返回文字说明。"
)

In [22]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="Hi, my name is 虎哥")]},
    config
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hi, my name is 虎哥
================================== Ai Message ==================================
Tool Calls:
  update_state (call_00_MqF3OHIWgKf3wpQ3laqC8960)
 Call ID: call_00_MqF3OHIWgKf3wpQ3laqC8960
  Args:
================================= Tool Message =================================
Name: update_state

Successfully updated agent state
================================== Ai Message ==================================

你好，虎哥！我是你的热心助手，很高兴认识你！有什么需要我帮忙的吗？尽管说，我会尽力帮你解决！😊


In [23]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi, my name is 虎哥', additional_kwargs={}, response_metadata={}, id='f8f968f2-5aef-4757-b097-c900b21a84b0'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 294, 'total_tokens': 321, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 38}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '48afcf3a-2502-4a37-9a65-3effe4cf7f0b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e01e5-184e-7fa2-b27c-04ef879917e0-0', tool_calls=[{'name': 'update_state', 'args': {}, 'id': 'call_00_MqF3OHIWgKf3wpQ3laqC8960', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 294, 'output_tokens': 27, 'total_tokens': 321, 'in

In [24]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我叫什么？")]},
    config
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hi, my name is 虎哥
================================== Ai Message ==================================
Tool Calls:
  update_state (call_00_MqF3OHIWgKf3wpQ3laqC8960)
 Call ID: call_00_MqF3OHIWgKf3wpQ3laqC8960
  Args:
================================= Tool Message =================================
Name: update_state

Successfully updated agent state
================================== Ai Message ==================================

你好，虎哥！我是你的热心助手，很高兴认识你！有什么需要我帮忙的吗？尽管说，我会尽力帮你解决！😊
================================ Human Message =================================

我叫什么？
================================== Ai Message ==================================
Tool Calls:
  update_state (call_00_A5KgeKyh1Gj1LlKfF0ft3927)
 Call ID: call_00_A5KgeKyh1Gj1LlKfF0ft3927
  Args:
================================= Tool Message =================================
Name: update_state

Successfully updated agent state
==========================

In [25]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi, my name is 虎哥', additional_kwargs={}, response_metadata={}, id='f8f968f2-5aef-4757-b097-c900b21a84b0'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 294, 'total_tokens': 321, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 38}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '48afcf3a-2502-4a37-9a65-3effe4cf7f0b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e01e5-184e-7fa2-b27c-04ef879917e0-0', tool_calls=[{'name': 'update_state', 'args': {}, 'id': 'call_00_MqF3OHIWgKf3wpQ3laqC8960', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 294, 'output_tokens': 27, 'total_tokens': 321, 'in

# 2. Store（长期记忆）

store是LangChain提供的长期记忆机制，用于在不同会话间共享数据。例如：模型以外的数据、用户偏好等。
LangChain提供了多种Store的实现方式，例如：
- InMemoryStore
- PostgresStore
- RedisStore
- ...

课程中我们以InMemoryStore为例来学习。


## 2.1 Store的数据结构

Store的数据格式是JSON文档，JSON文档采用分级管理：
- Namespace（命名空间）：可以理解为一个文件夹
  - Key（键）：可以理解为文件名，必须唯一
  - Value（值）：要存储的JSON文档

In [96]:
from langgraph.store.memory import InMemoryStore

# ==================== 1. 创建Store ====================
memory_store = InMemoryStore()


In [111]:

# ==================== 2.初始化一些数据 ====================
memory_store.put(
    ("preferences",),  # namespace，是一个tuple
    "user_001",  # key，可以是任意类型
    {  # value，是JSON格式文档
        "style": "business_markdown",
        "language": "zh-CN"
    }
)

memory_store.put(("preferences",), "user_002", {
    "style": "trump",
    "language": "en-US"
})

LangChain提供了查询store中的数据两种方式：
- get : 在指定namespace下根据key查找
- search : 在指定namespace下对value做语义搜索或者过滤


In [88]:
# ==================== 3.读取 ====================

# 3.1.基于get查询
user_preferences = memory_store.get(("preferences",), "user_001")
print(f"用户信息: {user_preferences.value if user_preferences else 'Not found'}")

# 3.2.基于search搜索数据
search_results = memory_store.search(
    ("preferences",),
    filter={"language": "zh-CN"},  # 基于字段做过滤查询
    limit=5
)
print(f"搜索结果数量: {len(search_results)}")
print(search_results)

用户信息: {'style': 'romantic', 'language': 'en-US'}
搜索结果数量: 1
[Item(namespace=['preferences'], key='user_002', value={'style': 'pirate', 'language': 'zh-CN'}, created_at='2026-04-22T14:57:06.381407+00:00', updated_at='2026-04-22T14:57:06.381407+00:00', score=None)]


## 2.2.基于向量模型的store

LangChain中的store支持基于向量相似度的语义检索，但需要设定一个向量模型。


In [97]:
from langgraph_cli.schemas import IndexConfig
from langchain_community.embeddings import DashScopeEmbeddings
import os

# ==================== 1. 创建Store ====================
# 初始化向量模型
embedding_model = DashScopeEmbeddings(
    model="text-embedding-v4", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 初始化store
memory_store = InMemoryStore(index=IndexConfig(
    embed=embedding_model,  # 向量模型
    dims=1024  # 向量维度
))

In [98]:
# ==================== 2.初始化一些数据 ====================
memory_store.put(("users",), "user_001", {
    "id": "user_001",
    "name": "张三",
    "department": "技术部",
    "clearance_level": 3
})

memory_store.put(("users",), "user_002", {
    "id": "user_002",
    "name": "李四",
    "department": "市场部",
    "clearance_level": 1
})

In [90]:
# ==================== 3.读取 ====================

# 3.1.基于get查询
user_data = memory_store.get(("users",), "user_001")
print(f"用户信息: {user_data.value if user_data else 'Not found'}")

# 3.2.基于search搜索数据
search_results = memory_store.search(
    ("users",),
    query="001",  # 基于字段语义检索
    limit=5
)
print(f"搜索结果数量: {len(search_results)}")
print(search_results)

用户信息: {'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}
搜索结果数量: 2
[Item(namespace=['users'], key='user_001', value={'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}, created_at='2026-04-22T14:57:14.414013+00:00', updated_at='2026-04-22T14:57:14.414016+00:00', score=0.3415490758450993), Item(namespace=['users'], key='user_002', value={'id': 'user_002', 'name': '李四', 'department': '市场部', 'clearance_level': 1}, created_at='2026-04-22T14:57:14.722126+00:00', updated_at='2026-04-22T14:57:14.722128+00:00', score=0.2891345529670016)]


## 2.3 在tool中访问store

In [56]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """获取用户信息"""
    if runtime.store is None:
        return "Store not available"

    user_info = runtime.store.get(("users",), user_id)

    if user_info is None:
        return "没有找到用户"

    return f"用户信息: {user_info.value}"

In [57]:
from langchain.messages import HumanMessage

# 在Agent中集成
agent = create_agent(
    model="deepseek-v4-flash",
    tools=[get_user_info],
    store=memory_store  # 指定store的存储方式
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询user_001的信息")]
})
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查询user_001的信息
================================== Ai Message ==================================

我来帮您查询用户 user_001 的信息。
Tool Calls:
  get_user_info (call_00_USeIOs56DPbMNu6ZyaEdCLKb)
 Call ID: call_00_USeIOs56DPbMNu6ZyaEdCLKb
  Args:
    user_id: user_001
================================= Tool Message =================================
Name: get_user_info

用户信息: {'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}
================================== Ai Message ==================================

根据查询结果，用户 user_001 的信息如下：

- **用户ID**: user_001
- **姓名**: 张三
- **部门**: 技术部
- **权限级别**: 3级

这是一个技术部的用户，拥有3级权限级别。


# 3. Context（运行时上下文）

Context用于在运行时传递配置参数、用户信息等会话临时数据。

## 3.1 定义Context Schema

In [58]:
from dataclasses import dataclass


# 方式1：使用dataclass定义Context
@dataclass
class UserContext:
    """Agent运行时上下文"""
    user_id: str = ""


# 方式2：使用TypedDict定义Context
from typing_extensions import TypedDict


class UserContext2(TypedDict):
    """运行时上下文类型"""
    user_id: str

## 3.2 在Tool中使用Context

In [113]:
from langchain.agents import create_agent

@tool
def get_users(runtime: ToolRuntime[UserContext]):
    """查询所有用户信息"""
    # 获取store
    store = runtime.store
    if store is None:
        return "Store not available"
    # 获取当前用户信息
    user_id = runtime.context.user_id
    if user_id is None:
        return "当前用户未登录，无法查看"

    user = store.get(("users",), user_id)
    if user is None:
        return "当前用户未登录，无法查看"

    # 校验权限，至少是3级权限
    user_info = dict(user.value)
    if user_info['clearance_level'] < 3:
        return "权限不足！"
    # 查询用户
    results = runtime.store.search(("users",))
    if results is None or len(results) == 0:
        return "未查询到用户"
    users = [item.value for item in results]
    return users


@tool
def get_user_preferences(runtime: ToolRuntime[UserContext]):
    """查询当前用户的偏好，根据偏好输出结果"""
    # 获取当前用户id
    user_id = runtime.context.user_id
    # 获取用户偏好
    user_preference = runtime.store.get(("preferences",), user_id)
    if user_preference is None:
        return "未查找到用户偏好信息"

    return user_preference.value


## 3.3.在Agent中使用context


In [114]:
# 创建Agent时指定context_schema
agent = create_agent(
    model="deepseek-v4-flash",
    tools=[get_users, get_user_preferences],
    store=memory_store,
    context_schema=UserContext,  # 指定Context类型
    system_prompt="""
    # indentify
    你是一个热心的助手，你可以调用工具获取用户信息，用户偏好。
    # instruction
    请务必按照用户偏好风格展示结果。
    """
)

In [115]:
# 调用时通过Context传递用户信息
response = agent.invoke(
    {"messages": [HumanMessage("Hello, 帮我查询所有用户信息")]},
    context=UserContext(user_id="user_001")
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hello, 帮我查询所有用户信息
================================== Ai Message ==================================

我来帮您查询所有用户信息。首先让我获取用户偏好，然后查询用户信息。
Tool Calls:
  get_user_preferences (call_00_rHSsTQeV1HnAR7rED4fgxzvX)
 Call ID: call_00_rHSsTQeV1HnAR7rED4fgxzvX
  Args:
================================= Tool Message =================================
Name: get_user_preferences

{"style": "business_markdown", "language": "zh-CN"}
================================== Ai Message ==================================

现在让我查询所有用户信息：
Tool Calls:
  get_users (call_00_4yCPc7HcsBEc8BHwnFLWWKIz)
 Call ID: call_00_4yCPc7HcsBEc8BHwnFLWWKIz
  Args:
================================= Tool Message =================================
Name: get_users

[{"id": "user_001", "name": "张三", "department": "技术部", "clearance_level": 3}, {"id": "user_002", "name": "李四", "department": "市场部", "clearance_level": 1}]
================================== Ai Messa

# 4. 总结

1. **State（短期记忆）**
   - 通过`runtime.state`参数访问
   - 用于保存会话历史消息、对话状态、计数等临时信息
   - 生命周期是当前会话
   - 存储方式：可以是InMemorySaver、也可以是数据库


2. **Store（长期记忆）**
   - 通过`runtime.store`访问
   - 用于拓展知识、用户偏好等信息
   - 生命周期跨越多个会话
   - 支持InMemoryStore和数据库存储


3. **Context（运行时上下文）**
   - 通过`runtime.context`访问
   - 用于传递用户ID、配置参数等
   - 生命周期是当前会话
   - 存储方式：基于内存


---

**更多资源**:
- [LangChain官方文档 - Runtime](https://docs.langchain.com/oss/python/langchain/runtime)
- [LangChain官方文档 - Long-term Memory](https://docs.langchain.com/oss/python/langchain/long-term-memory)
- [LangGraph Store文档](https://langchain-ai.github.io/langgraph/concepts/store/)